# Viraltest v2 — TRL GRPO Training

End-to-end pipeline:
1. **Episode collection** — calls `inference.py::collect_episode` against the running Viraltest env using ANY OpenAI-compatible LLM endpoint.
   - Local Gemma (free): `API_BASE_URL=http://0.0.0.0:1337/v1`
   - HF Inference Router (cheap): `API_BASE_URL=https://router.huggingface.co/v1`, `MODEL_NAME=Qwen/Qwen2.5-1.5B-Instruct`
2. **Baseline vs agent comparison** — reuses `scripts/run_baseline_vs_agent.py` data.
3. **GRPO weight updates** — only when `RUN_REAL_GRPO=1`. Requires Colab T4 / GPU + transformers; not runnable on llama.cpp.
4. **Plots** — calls `scripts/make_plots.py` so the notebook and README stay in sync.

Reward = per-step env reward + 2 × terminal `grader_score`.

In [ ]:
import os, sys, json
from pathlib import Path

ROOT = Path('..').resolve() if Path('inference.py').exists() is False else Path('.').resolve()
sys.path.insert(0, str(ROOT))

RUN_REAL_GRPO = os.getenv('RUN_REAL_GRPO', '0') == '1'
ENV_BASE_URL = os.getenv('ENV_BASE_URL', 'http://localhost:8000')
API_BASE_URL = os.getenv('API_BASE_URL', 'http://0.0.0.0:1337/v1')
TASKS = ['monthly_engage', 'monthly_strategic', 'monthly_competitive']

print(f'ROOT          = {ROOT}')
print(f'ENV_BASE_URL  = {ENV_BASE_URL}')
print(f'API_BASE_URL  = {API_BASE_URL}')
print(f'RUN_REAL_GRPO = {RUN_REAL_GRPO}')

In [ ]:
# Episode collection (no token cost when API_BASE_URL is localhost)
import asyncio
from inference import collect_episode, make_client, MODEL_NAME

async def collect_for(arm_use_tools: bool, tasks=TASKS, episodes=1, max_steps=30):
    client = make_client() if arm_use_tools else None
    out = []
    for task in tasks:
        for ep in range(episodes):
            r = await collect_episode(task, use_tools=arm_use_tools, client=client,
                                      model=MODEL_NAME, max_steps=max_steps, quiet=True)
            print(f"  {('agent' if arm_use_tools else 'baseline')} {task} ep{ep+1}: "
                  f"reward_sum={sum(r['rewards']):.3f} grader={r['grader_score']:.3f}")
            out.append(r)
    return out

print('Collecting baseline (heuristic, no LLM call) ...')
baseline = asyncio.run(collect_for(arm_use_tools=False, episodes=3))
print('Collecting agent (Gemma + tools) ...')
agent = asyncio.run(collect_for(arm_use_tools=True, episodes=1))

In [ ]:
import statistics

for task in TASKS:
    b = [r['grader_score'] for r in baseline if r['task'] == task]
    a = [r['grader_score'] for r in agent if r['task'] == task]
    print(f'{task:24s}  baseline_mean={statistics.mean(b):.3f}  agent_mean={statistics.mean(a):.3f}'
          f'  delta={statistics.mean(a)-statistics.mean(b):+.3f}')

In [ ]:
# Persist trajectories to runs/*.jsonl so make_plots.py can render them
(ROOT / 'runs').mkdir(exist_ok=True)
with (ROOT / 'runs' / 'baseline.jsonl').open('w') as f:
    for r in baseline: f.write(json.dumps(r) + '\n')
with (ROOT / 'runs' / 'agent.jsonl').open('w') as f:
    for r in agent: f.write(json.dumps(r) + '\n')
print(f'wrote {ROOT}/runs/baseline.jsonl ({len(baseline)})  agent.jsonl ({len(agent)})')

import subprocess
subprocess.run([sys.executable, str(ROOT / 'scripts' / 'make_plots.py')], check=True)

## Real GRPO weight updates (Colab T4 / GPU only)

The cells below only run when `RUN_REAL_GRPO=1` is set. They require:
- A GPU runtime (T4 free tier on Colab is enough for Qwen2.5-1.5B-Instruct).
- `pip install -q trl transformers accelerate peft bitsandbytes`.
- The Viraltest env reachable at `ENV_BASE_URL`.

Local llama.cpp Gemma cannot do gradient updates — we use it for episode rollouts only. Switch `MODEL_NAME` to a HF-hosted base model for the GRPO step.

In [ ]:
if not RUN_REAL_GRPO:
    print('RUN_REAL_GRPO=0 — skipping weight-update cells.')
    print('Set RUN_REAL_GRPO=1 to enable on a GPU runtime.')
else:
    !pip install -q trl transformers accelerate peft bitsandbytes
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from trl import GRPOConfig, GRPOTrainer

    BASE_MODEL = os.getenv('BASE_MODEL', 'Qwen/Qwen2.5-1.5B-Instruct')
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, trust_remote_code=True,
                                                 torch_dtype='auto')

    def reward_fn(prompts, completions, **_):
        # Map each (prompt, completion) to env reward via env.step()
        # The collect_episode helper above shows the expected JSON shape.
        # For brevity here we score by terminal grader; expand for per-step rewards.
        scores = []
        for completion in completions:
            # TODO: parse completion JSON → ViraltestAction → env.step → grader_score
            scores.append(0.0)
        return scores

    cfg = GRPOConfig(output_dir='./viraltest-grpo', num_train_epochs=1,
                     per_device_train_batch_size=2, learning_rate=5e-6,
                     report_to='wandb' if os.getenv('WANDB_API_KEY') else 'none')
    trainer = GRPOTrainer(model=model, args=cfg, reward_funcs=[reward_fn],
                          train_dataset=None, processing_class=tokenizer)
    trainer.train()